# Manual Nuclei Counter
### Interactive click-to-count tool for Leica `.lif` files
**Workflow:**
1. Edit the **CONFIG** cell below
2. Run **all cells** (`Kernel → Restart & Run All`)
3. The counter window opens — click nuclei, navigate series, save results

---
**Controls:**
| Action | How |
|--------|-----|
| Count a nucleus | Left-click on it |
| Remove last click | `Undo` button |
| Clear current series | `Reset` button |
| Go to previous/next series | `< Prev` / `Next >` buttons |
| Export CSV + Excel | `Save` button (can click multiple times) |
| Adjust brightness/contrast | Drag the **Black pt** / **White pt** sliders (instant response) |

> **Cellpose** runs automatically in a background thread for each series (protoplast count, C=2).  
> The title bar updates once it finishes — no need to wait before clicking nuclei.

## 1 · Configuration

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                        USER CONFIG                              ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Input files ──────────────────────────────────────────────────────────────
# Option A: point to a folder — all .lif files inside will be loaded
LIF_FOLDER = "/Users/mark/Desktop/PVC Revision Experiments/20251215_fluorophoreDelivery/18hours_c3c18"  # set to None to use LIF_PATHS instead

# Option B: explicit list of files (used only when LIF_FOLDER is None)
LIF_PATHS = [
    # "/path/to/another.lif",
]

# ── Resolve final file list (do not edit) ────────────────────────────────────
from pathlib import Path as _Path
if LIF_FOLDER is not None:
    _folder = _Path(LIF_FOLDER)
    LIF_PATHS = sorted(_folder.glob("*.lif"))
    if not LIF_PATHS:
        raise FileNotFoundError(f"No .lif files found in: {_folder}")
    print(f"Found {len(LIF_PATHS)} .lif file(s) in {_folder}:")
    for p in LIF_PATHS:
        print(f"  {p.name}")
else:
    LIF_PATHS = [_Path(p) for p in LIF_PATHS]
    print(f"Using {len(LIF_PATHS)} manually specified .lif file(s).")

# ── Output directory ─────────────────────────────────────────────────────────
OUTPUT_DIR = "./18hours_c3c18-counted"

# ── Channel indices (0-based) ─────────────────────────────────────────────────
CHANNEL_NUCLEI = 1   # channel to display for manual counting (e.g. DAPI / GFP nuclei)
CHANNEL_CYTO   = 2   # channel for automated Cellpose protoplast count

# ── Z-stack handling ──────────────────────────────────────────────────────────
USE_MAX_PROJECTION = True   # True = max-project z-stacks; False = use first z-slice

# ── Cellpose (automated protoplast count) ─────────────────────────────────────
RUN_CELLPOSE = True          # set False to skip automated counting entirely
CELLPOSE_PARAMS = {
    # model_type removed in Cellpose v4 — cpsam is the single universal model
    "diameter":           None,    # None = auto-estimate; set e.g. 80 if you know it
    "flow_threshold":     0.4,
    "cellprob_threshold": 0.0,
    "min_size":           50,
}

# ── QC overlays ───────────────────────────────────────────────────────────────────
SAVE_QC = True   # save side-by-side raw/mask PNG per series to OUTPUT_DIR/cellpose_qc/

Found 3 .lif file(s) in /Users/mark/Desktop/PVC Revision Experiments/20251215_fluorophoreDelivery/18hours_c3c18:
  empty-c3c18-1.lif
  empty-c3c18-2.lif
  empty-c3c18-3.lif


## 2 · Imports

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from readlif.reader import LifFile
from PIL import Image

# Matplotlib — use interactive backend
import matplotlib
matplotlib.use("TkAgg")          # change to "Qt5Agg" if TkAgg is unavailable
import matplotlib.pyplot as plt
import matplotlib.widgets as mwidgets
from matplotlib.gridspec import GridSpec

print("Imports OK")

Imports OK


## 3 · Helper Functions

In [3]:
# ── Helper functions ──────────────────────────────────────────────────────────

def extract_channel(img, channel_idx, z_idx=0):
    """Extract a single 2-D plane from a LIF image object."""
    try:
        frame = img.get_frame(z=z_idx, t=0, c=channel_idx)
        return np.array(frame, dtype=np.float32)
    except Exception:
        return np.zeros((img.dims.y, img.dims.x), dtype=np.float32)

def extract_max_projection(img, channel_idx):
    """Max-project all z-slices for a given channel."""
    planes = []
    for z in range(img.dims.z):
        planes.append(extract_channel(img, channel_idx, z_idx=z))
    return np.max(np.stack(planes, axis=0), axis=0) if planes else np.zeros(
        (img.dims.y, img.dims.x), dtype=np.float32)

def normalize_to_uint8(arr):
    """Stretch to [0, 255] uint8."""
    a = arr.astype(np.float32)
    lo, hi = a.min(), a.max()
    if hi == lo:
        return np.zeros_like(a, dtype=np.uint8)
    return ((a - lo) / (hi - lo) * 255).astype(np.uint8)

print("Helper functions defined")

Helper functions defined


## 4 · ManualNucleiCounter Class

In [4]:
class ManualNucleiCounter:
    """
    Interactive manual nuclei counter.
    - B/C sliders operate in raw pixel intensity space for uniform sensitivity.
    - TextBox widgets allow typing values directly.
    - Cellpose runs in a single background thread (one series at a time) so the
      GUI stays responsive without spawning subprocesses.  ProcessPoolExecutor
      was removed because Cellpose v4 (CPSAM) uses PyTorch sparse tensors, and
      multiple spawned processes initialising PyTorch simultaneously on macOS /
      Apple Silicon triggers a SEGFAULT.  A single thread is crash-safe and
      sufficient given the interactive pace of manual counting.
    - Close-safety: draw_idle() is never called after window close.
    """

    def __init__(self, lif_paths, output_dir,
                 channel_nuclei=0, channel_cyto=2,
                 use_max_projection=True,
                 run_cellpose=True,
                 cellpose_params=None):
        self.lif_paths          = [Path(p) for p in (lif_paths if isinstance(lif_paths, list) else [lif_paths])]
        self.output_dir         = Path(output_dir)
        self.channel_nuclei     = channel_nuclei
        self.channel_cyto       = channel_cyto
        self.use_max_projection = use_max_projection
        self.run_cellpose       = run_cellpose
        self.series_list = self._enumerate_series()
        self.n_series    = len(self.series_list)
        self.current_idx = 0
        self.state = {s["key"]: {"clicks": [], "cyto_count": None} for s in self.series_list}
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # GUI state
        self.fig          = None
        self.ax_img       = None
        self._im          = None
        self._markers     = []
        self._loading     = False
        self._tb_updating = False
        self._cid_click   = None
        self._window_open = False

        # Threading
        import threading, queue
        self._save_lock  = threading.Lock()
        self._cp_queue   = queue.Queue()
        self._cp_worker  = None

        # Strip non-Cellpose keys from params
        _cp = dict(cellpose_params) if cellpose_params else {}
        _cp.pop('num_workers', None)          # no longer used
        self.save_qc         = _cp.pop('save_qc', SAVE_QC)
        self.cellpose_params = _cp

    # ── Series enumeration ────────────────────────────────────────────────────

    def _enumerate_series(self):
        series_list = []
        for lif_path in self.lif_paths:
            lif = LifFile(str(lif_path))
            for idx, img in enumerate(lif.get_iter_image()):
                key = f"{lif_path.stem}__{idx}__{img.name}"
                series_list.append({
                    "key": key, "lif_path": lif_path,
                    "index": idx, "name": img.name, "lif_name": lif_path.name,
                })
        return series_list

    # ── Image loading ─────────────────────────────────────────────────────────

    def _load_series_images(self, series_info):
        lif = LifFile(str(series_info["lif_path"]))
        img = lif.get_image(series_info["index"])
        if self.use_max_projection and img.dims.z > 1:
            nuclei_raw = extract_max_projection(img, self.channel_nuclei)
            cyto_raw   = extract_max_projection(img, self.channel_cyto)
        else:
            nuclei_raw = extract_channel(img, self.channel_nuclei)
            cyto_raw   = extract_channel(img, self.channel_cyto)
        return nuclei_raw.astype(np.float32), cyto_raw

    # ── Cellpose worker (single background thread) ────────────────────────────

    def _cellpose_worker(self):
        """
        Runs in one background thread.  Processes the queue one series at a
        time — safe on macOS because PyTorch never needs to initialise in a
        spawned subprocess (which is what caused the SEGFAULT with CPSAM).
        The model is loaded once and reused for every series.
        """
        from cellpose import models as _cp_models
        cp_model = _cp_models.CellposeModel(gpu=False)   # load once, reuse
        done = 0

        while True:
            job = self._cp_queue.get()
            if job is None:              # shutdown sentinel
                self._cp_queue.task_done()
                break

            series_key, cyto_raw = job
            series_name = next(
                (s["name"] for s in self.series_list if s["key"] == series_key),
                series_key)

            try:
                a       = cyto_raw.astype(np.float32)
                lo, hi  = float(a.min()), float(a.max())
                cyto_u8 = (np.zeros_like(a, dtype=np.uint8) if hi == lo
                           else ((a - lo) / (hi - lo) * 255).astype(np.uint8))
                del cyto_raw, a

                masks, _, _ = cp_model.eval(
                    cyto_u8,
                    diameter           = self.cellpose_params.get("diameter"),
                    flow_threshold     = self.cellpose_params.get("flow_threshold",     0.4),
                    cellprob_threshold = self.cellpose_params.get("cellprob_threshold", 0.0),
                    min_size           = self.cellpose_params.get("min_size", 50),
                )
                count = int(masks.max())
                self.state[series_key]["cyto_count"] = count
                done += 1

                qc_path = None
                if self.save_qc:
                    s_info  = next(s for s in self.series_list if s["key"] == series_key)
                    qc_dir  = self.output_dir / s_info["lif_path"].stem / "cellpose_qc"
                    qc_dir.mkdir(parents=True, exist_ok=True)
                    qc_path = self._write_qc_png(cyto_u8, masks, series_name, qc_dir)

                qc_msg = f"  QC -> {Path(qc_path).name}" if qc_path else ""
                print(f"  [Cellpose {done}] Done: {series_name} -> {count} cells{qc_msg}",
                      flush=True)

            except Exception as e:
                done += 1
                self.state[series_key]["cyto_count"] = -1
                print(f"  [Cellpose {done}] Warning [{series_name}]: {e}", flush=True)

            if self._window_open and self.fig is not None:
                try:
                    self.fig.canvas.draw_idle()
                except Exception:
                    pass

            self._cp_queue.task_done()

    @staticmethod
    def _write_qc_png(cyto_u8, masks, series_name, qc_dir):
        """Save a side-by-side raw/mask QC image. Returns path string."""
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from cellpose.utils import masks_to_outlines

        n_masks = masks.max()
        rng     = np.random.default_rng(seed=42)
        colors  = rng.uniform(0.3, 1.0, size=(n_masks + 1, 3))
        colors[0] = 0
        overlay = np.zeros((*masks.shape, 4), dtype=np.float32)
        for lbl in range(1, n_masks + 1):
            px = masks == lbl
            overlay[px, :3] = colors[lbl]
            overlay[px,  3] = 0.45
        outlines = masks_to_outlines(masks)
        overlay[outlines, :3] = 1.0
        overlay[outlines,  3] = 0.9

        fig, axes = plt.subplots(1, 2, figsize=(12, 6), facecolor="#111111",
                                 gridspec_kw={"wspace": 0.04})
        for ax in axes:
            ax.set_facecolor("#111111")
            ax.axis("off")
        axes[0].imshow(cyto_u8, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[0].set_title("Raw (auto-contrast)", color="white", fontsize=10)
        axes[1].imshow(cyto_u8, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[1].imshow(overlay, interpolation="nearest")
        axes[1].set_title(f"Cellpose masks  ({n_masks} cells)", color="white", fontsize=10)
        fig.suptitle(series_name, color="white", fontsize=11, y=1.01)

        safe = "".join(c if c.isalnum() or c in "-_ " else "_" for c in series_name).strip()
        out  = qc_dir / f"{safe}_cellpose_qc.png"
        fig.savefig(str(out), dpi=150, bbox_inches="tight", facecolor="#111111")
        plt.close(fig)
        return str(out)

    def _enqueue_cellpose(self, cyto_raw, series_key):
        """Add a job to the queue. Start the worker thread if not running."""
        import threading
        if self._cp_worker is None or not self._cp_worker.is_alive():
            self._cp_worker = threading.Thread(
                target=self._cellpose_worker, daemon=False, name="CellposeWorker")
            self._cp_worker.start()
        self._cp_queue.put((series_key, cyto_raw))

    # ── B/C controls ──────────────────────────────────────────────────────────

    def _apply_clim(self):
        if self._im is None:
            return
        lo = self._slider_vmin.val
        hi = self._slider_vmax.val
        if hi <= lo:
            hi = lo + 1.0
        self._im.set_clim(lo, hi)
        self.fig.canvas.draw_idle()

    def _on_bc_changed(self, _val):
        if self._loading or self._im is None:
            return
        self._tb_updating = True
        self._tb_vmin.set_val(f"{self._slider_vmin.val:.0f}")
        self._tb_vmax.set_val(f"{self._slider_vmax.val:.0f}")
        self._tb_updating = False
        self._apply_clim()

    def _on_tb_vmin_submit(self, text):
        if self._tb_updating or self._loading:
            return
        raw = self._current_nuclei_raw
        img_min, img_max = float(raw.min()), float(raw.max())
        try:
            v = max(img_min, min(float(text), img_max))
        except ValueError:
            v = self._slider_vmin.val
        self._tb_updating = True
        self._slider_vmin.set_val(v)
        self._tb_updating = False

    def _on_tb_vmax_submit(self, text):
        if self._tb_updating or self._loading:
            return
        raw = self._current_nuclei_raw
        img_min, img_max = float(raw.min()), float(raw.max())
        try:
            v = max(img_min, min(float(text), img_max))
        except ValueError:
            v = self._slider_vmax.val
        self._tb_updating = True
        self._slider_vmax.set_val(v)
        self._tb_updating = False

    # ── Redraw markers + title ────────────────────────────────────────────────

    def _redraw_markers(self):
        for artist in self._markers:
            artist.remove()
        self._markers = []

        s      = self.series_list[self.current_idx]
        key    = s["key"]
        clicks = self.state[key]["clicks"]

        for i, (cx, cy) in enumerate(clicks):
            dot, = self.ax_img.plot(
                cx, cy, "o", color="#E9ED4C", markersize=8,
                markeredgecolor="black", markeredgewidth=0.8, zorder=5)
            lbl = self.ax_img.text(
                cx + 6, cy - 6, str(i + 1),
                color="#E9ED4C", fontsize=7, fontweight="bold", zorder=6)
            self._markers.extend([dot, lbl])

        cyto_count = self.state[key]["cyto_count"]
        cyto_str   = (str(cyto_count) if (cyto_count is not None and cyto_count >= 0)
                      else ("running..." if cyto_count is None else "failed"))
        npc = (len(clicks) / cyto_count) if (cyto_count and cyto_count > 0) else float("nan")
        npc_str = f"{npc:.2f}" if not np.isnan(npc) else "N/A"

        title = (
            f"[{self.current_idx + 1}/{self.n_series}]  {s['name']}  —  {s['lif_name']}\n"
            f"Nuclei (manual): {len(clicks)}   |   "
            f"Protoplasts (Cellpose): {cyto_str}   |   "
            f"Nuclei/cell: {npc_str}"
        )
        self.ax_img.set_title(title, color="white", fontsize=10, pad=6)
        self.fig.canvas.draw_idle()

    # ── Load a new series ─────────────────────────────────────────────────────

    def _load_and_show(self):
        self._loading = True
        s   = self.series_list[self.current_idx]
        key = s["key"]
        print(f"Loading [{self.current_idx + 1}/{self.n_series}]: {s['name']} ...")

        nuclei_raw, cyto_raw = self._load_series_images(s)
        self._current_nuclei_raw = nuclei_raw

        img_min = float(nuclei_raw.min())
        img_max = float(nuclei_raw.max())
        lo_init = float(np.percentile(nuclei_raw, 1))
        hi_init = float(np.percentile(nuclei_raw, 99))

        self._slider_vmin.valmin = img_min
        self._slider_vmin.valmax = img_max
        self._slider_vmin.ax.set_xlim(img_min, img_max)
        self._slider_vmax.valmin = img_min
        self._slider_vmax.valmax = img_max
        self._slider_vmax.ax.set_xlim(img_min, img_max)

        self._slider_vmin.set_val(lo_init)
        self._slider_vmax.set_val(hi_init)

        self._tb_updating = True
        self._tb_vmin.set_val(f"{lo_init:.0f}")
        self._tb_vmax.set_val(f"{hi_init:.0f}")
        self._tb_updating = False

        if self._im is None:
            self._im = self.ax_img.imshow(
                nuclei_raw, cmap="gray", vmin=lo_init, vmax=hi_init,
                interpolation="nearest", aspect="equal")
            self.ax_img.set_facecolor("#111111")
            self.ax_img.axis("off")
        else:
            self._im.set_data(nuclei_raw)
            self._im.set_clim(lo_init, hi_init)
            h, w = nuclei_raw.shape
            self.ax_img.set_xlim(-0.5, w - 0.5)
            self.ax_img.set_ylim(h - 0.5, -0.5)

        self._loading = False

        if self.run_cellpose and self.state[key]["cyto_count"] is None:
            self._enqueue_cellpose(cyto_raw, key)
        else:
            del cyto_raw

        self._redraw_markers()

    # ── Event handlers ────────────────────────────────────────────────────────

    def _on_click(self, event):
        if event.inaxes != self.ax_img or event.button != 1:
            return
        key = self.series_list[self.current_idx]["key"]
        self.state[key]["clicks"].append((event.xdata, event.ydata))
        self._redraw_markers()

    def _on_undo(self, event):
        key = self.series_list[self.current_idx]["key"]
        if self.state[key]["clicks"]:
            self.state[key]["clicks"].pop()
            self._redraw_markers()

    def _on_reset(self, event):
        key = self.series_list[self.current_idx]["key"]
        self.state[key]["clicks"] = []
        self._redraw_markers()

    def _on_prev(self, event):
        if self.current_idx > 0:
            self.current_idx -= 1
            self._load_and_show()

    def _on_next(self, event):
        if self.current_idx < self.n_series - 1:
            self.current_idx += 1
            self._load_and_show()

    def _on_save(self, event):
        import threading
        threading.Thread(target=self._export_results, daemon=True).start()

    # ── Export ────────────────────────────────────────────────────────────────

    def _export_results(self):
        """
        Writes results per LIF file into OUTPUT_DIR/<lif_stem>/
          - manual_counts.csv
          - manual_counts.xlsx  (data sheet + summary stats sheet)
        Also writes a combined summary across all files:
          - OUTPUT_DIR/all_counts.csv
          - OUTPUT_DIR/all_counts.xlsx
        """
        import shutil, tempfile
        if not self._save_lock.acquire(blocking=False):
            print("Save already in progress, please wait...")
            return
        try:
            snapshot = {
                s["key"]: {
                    "clicks":     list(self.state[s["key"]]["clicks"]),
                    "cyto_count": self.state[s["key"]]["cyto_count"],
                }
                for s in self.series_list
            }

            records = []
            for s in self.series_list:
                key    = s["key"]
                clicks = snapshot[key]["clicks"]
                cyto_n = snapshot[key]["cyto_count"]
                nuc_n  = len(clicks)
                npc    = (nuc_n / cyto_n) if (cyto_n and cyto_n > 0) else None
                records.append({
                    "lif_file":            s["lif_name"],
                    "lif_stem":            s["lif_path"].stem,
                    "series_index":        s["index"],
                    "series_name":         s["name"],
                    "protoplast_count":    cyto_n if (cyto_n is not None and cyto_n >= 0) else None,
                    "nuclei_count_manual": nuc_n,
                    "nuclei_per_cell":     round(npc, 3) if npc is not None else None,
                    "click_coordinates":   str(clicks),
                })

            all_df = pd.DataFrame(records)

            for lif_stem, lif_df in all_df.groupby("lif_stem"):
                lif_dir = self.output_dir / lif_stem
                lif_dir.mkdir(parents=True, exist_ok=True)
                out_df   = lif_df.drop(columns=["lif_stem"]).reset_index(drop=True)
                csv_path = lif_dir / "manual_counts.csv"
                out_df.to_csv(csv_path, index=False)
                tmp = Path(tempfile.mktemp(suffix=".xlsx"))
                with pd.ExcelWriter(str(tmp), engine="openpyxl") as w:
                    out_df.to_excel(w, sheet_name="Manual_Counts", index=False)
                    ok = out_df[out_df["nuclei_count_manual"] > 0]
                    if not ok.empty:
                        ok[["protoplast_count", "nuclei_count_manual", "nuclei_per_cell"]] \
                          .describe().to_excel(w, sheet_name="Summary_Stats")
                shutil.copy(str(tmp), str(lif_dir / "manual_counts.xlsx"))
                tmp.unlink(missing_ok=True)
                print(f"  [{lif_stem}] Saved: {csv_path}", flush=True)

            combined_csv = self.output_dir / "all_counts.csv"
            all_df.drop(columns=["lif_stem"]).to_csv(combined_csv, index=False)
            tmp = Path(tempfile.mktemp(suffix=".xlsx"))
            with pd.ExcelWriter(str(tmp), engine="openpyxl") as w:
                all_df.drop(columns=["lif_stem"]).to_excel(w, sheet_name="All_Counts", index=False)
                ok = all_df[all_df["nuclei_count_manual"] > 0]
                if not ok.empty:
                    ok[["protoplast_count", "nuclei_count_manual", "nuclei_per_cell"]] \
                      .describe().to_excel(w, sheet_name="Summary_Stats")
                per_lif = all_df.groupby("lif_stem").agg(
                    series_count=("series_name", "count"),
                    total_protoplasts=("protoplast_count", "sum"),
                    total_nuclei=("nuclei_count_manual", "sum"),
                    mean_nuclei_per_cell=("nuclei_per_cell", "mean"),
                ).reset_index()
                per_lif.to_excel(w, sheet_name="Per_LIF_Summary", index=False)
            shutil.copy(str(tmp), str(self.output_dir / "all_counts.xlsx"))
            tmp.unlink(missing_ok=True)

            print(f"Combined: {combined_csv}", flush=True)
            print(f"Combined: {self.output_dir / 'all_counts.xlsx'}", flush=True)
            print(all_df[["lif_file", "series_name", "protoplast_count",
                           "nuclei_count_manual", "nuclei_per_cell"]]
                  .to_string(index=False))
            return all_df
        finally:
            self._save_lock.release()

    # ── Launch ────────────────────────────────────────────────────────────────

    def launch(self):
        import matplotlib.pyplot as plt
        import matplotlib.widgets as mwidgets
        from matplotlib.gridspec import GridSpec

        plt.rcParams["toolbar"] = "None"

        self.fig = plt.figure(figsize=(10, 10), facecolor="#1a1a1a")
        gs = GridSpec(4, 1, figure=self.fig,
                      height_ratios=[14, 0.6, 0.6, 1.2], hspace=0.08)

        self.ax_img = self.fig.add_subplot(gs[0])
        ax_vmin     = self.fig.add_subplot(gs[1])
        ax_vmax     = self.fig.add_subplot(gs[2])
        ax_buttons  = self.fig.add_subplot(gs[3])
        ax_buttons.axis("off")
        self.ax_img.set_facecolor("#111111")

        self._slider_vmin = mwidgets.Slider(
            ax_vmin, "Black pt", 0, 1, valinit=0,
            color="#555555", track_color="#333333")
        self._slider_vmax = mwidgets.Slider(
            ax_vmax, "White pt", 0, 1, valinit=1,
            color="#AAAAAA", track_color="#333333")
        for sl in (self._slider_vmin, self._slider_vmax):
            sl.label.set_color("white")
            sl.valtext.set_visible(False)

        self._slider_vmin.on_changed(self._on_bc_changed)
        self._slider_vmax.on_changed(self._on_bc_changed)

        ax_tb_vmin = self.fig.add_axes([0.88, 0.148, 0.09, 0.030])
        ax_tb_vmax = self.fig.add_axes([0.88, 0.108, 0.09, 0.030])
        self._tb_vmin = mwidgets.TextBox(ax_tb_vmin, "", initial="0",
                                         color="#2a2a2a", hovercolor="#3a3a3a")
        self._tb_vmax = mwidgets.TextBox(ax_tb_vmax, "", initial="1",
                                         color="#2a2a2a", hovercolor="#3a3a3a")
        for tb in (self._tb_vmin, self._tb_vmax):
            tb.text_disp.set_color("white")
            tb.text_disp.set_fontsize(8)

        self._tb_vmin.on_submit(self._on_tb_vmin_submit)
        self._tb_vmax.on_submit(self._on_tb_vmax_submit)

        self.fig.text(0.925, 0.182, "Black pt", color="#AAAAAA",
                      fontsize=7, ha="center", va="bottom")
        self.fig.text(0.925, 0.142, "White pt", color="#AAAAAA",
                      fontsize=7, ha="center", va="bottom")

        btn_specs = [
            ("< Prev", 0.01, 0.10, self._on_prev,  "#333333"),
            ("Next >", 0.12, 0.10, self._on_next,  "#333333"),
            ("Undo",   0.30, 0.10, self._on_undo,  "#FF9400"),
            ("Reset",  0.41, 0.10, self._on_reset, "#CC3333"),
            ("Save",   0.80, 0.18, self._on_save,  "#75A025"),
        ]
        self._buttons = []
        for label, x, w, cb, color in btn_specs:
            ax_b = self.fig.add_axes([x, 0.01, w, 0.04])
            btn  = mwidgets.Button(ax_b, label, color=color, hovercolor="#FFFFFF")
            btn.label.set_color("white")
            btn.label.set_fontsize(9)
            btn.on_clicked(cb)
            self._buttons.append(btn)

        self.fig.text(
            0.5, 0.001,
            "LEFT CLICK = add nucleus   |   Undo = remove last   |   "
            "Reset = clear series   |   Save = export CSV/Excel",
            ha="center", va="bottom", color="#AAAAAA", fontsize=8)

        self._cid_click   = self.fig.canvas.mpl_connect("button_press_event", self._on_click)
        self._window_open = True
        self._load_and_show()
        plt.show()                    # blocks until window is closed
        self._window_open = False

        # Post-close: drain the queue, then final export
        if self.run_cellpose and self._cp_worker is not None and self._cp_worker.is_alive():
            remaining = self._cp_queue.qsize()
            if remaining > 0:
                print(f"\nWindow closed. Cellpose still has {remaining} series queued — "
                      f"running in background (do not interrupt the kernel)...")
            self._cp_queue.put(None)
            self._cp_worker.join()
            print("Cellpose complete.")

        print("Saving final results...")
        self._export_results()

## 5 · Launch Counter

In [5]:
# ── Launch the counter ────────────────────────────────────────────────────────
# Run this cell to open the interactive window.
# The window is blocking — close it (or press Ctrl+C here) when done.

# Quick summary before opening
from readlif.reader import LifFile as _LifFile
_total_series = sum(sum(1 for _ in _LifFile(str(p)).get_iter_image()) for p in LIF_PATHS)
print(f"Files: {len(LIF_PATHS)}  |  Total series: {_total_series}")
print("Files to process:")
for _p in LIF_PATHS:
    _n = sum(1 for _ in _LifFile(str(_p)).get_iter_image())
    print(f"  {_p.name}  ({_n} series)")

CELLPOSE_PARAMS['save_qc'] = SAVE_QC

counter = ManualNucleiCounter(
    lif_paths          = LIF_PATHS,
    output_dir         = OUTPUT_DIR,
    channel_nuclei     = CHANNEL_NUCLEI,
    channel_cyto       = CHANNEL_CYTO,
    use_max_projection = USE_MAX_PROJECTION,
    run_cellpose       = RUN_CELLPOSE,
    cellpose_params    = CELLPOSE_PARAMS,
)
counter.launch()

Files: 3  |  Total series: 30
Files to process:
  empty-c3c18-1.lif  (10 series)
  empty-c3c18-2.lif  (10 series)
  empty-c3c18-3.lif  (10 series)
Loading [1/30]: Image001 ...
Loading [2/30]: Image002 ...
Loading [3/30]: Image003 ...
Loading [4/30]: Image004 ...
Loading [3/30]: Image003 ...
Loading [4/30]: Image004 ...
Loading [5/30]: Image005 ...
Loading [6/30]: Image006 ...
Loading [7/30]: Image007 ...
Loading [8/30]: Image008 ...
Loading [9/30]: Image009 ...
Loading [10/30]: Image010 ...
Loading [11/30]: Image001 ...
Loading [12/30]: Image002 ...
Loading [13/30]: Image003 ...
Loading [14/30]: Image004 ...
Loading [15/30]: Image005 ...
Loading [16/30]: Image006 ...
Loading [17/30]: Image007 ...
Loading [18/30]: Image008 ...
Loading [19/30]: Image009 ...
Loading [20/30]: Image010 ...
Loading [19/30]: Image009 ...
Loading [20/30]: Image010 ...
Loading [21/30]: Image001 ...
Loading [22/30]: Image002 ...
Loading [23/30]: Image003 ...
Loading [24/30]: Image004 ...
Loading [25/30]: Image00

## 6 · Review / Re-export Results

In [6]:
# ── Review / re-export results after closing the window ──────────────────────
# Run this cell at any time to see current counts and re-save.

if "counter" in dir():
    df = counter._export_results()
    display(df)
else:
    print("Run the launch cell first.")

  [empty-c3c18-1] Saved: 18hours_c3c18-counted/empty-c3c18-1/manual_counts.csv
  [empty-c3c18-2] Saved: 18hours_c3c18-counted/empty-c3c18-2/manual_counts.csv
  [empty-c3c18-3] Saved: 18hours_c3c18-counted/empty-c3c18-3/manual_counts.csv
Combined: 18hours_c3c18-counted/all_counts.csv
Combined: 18hours_c3c18-counted/all_counts.xlsx
         lif_file series_name  protoplast_count  nuclei_count_manual  nuclei_per_cell
empty-c3c18-1.lif    Image001               171                    0            0.000
empty-c3c18-1.lif    Image002               220                    0            0.000
empty-c3c18-1.lif    Image003               250                    0            0.000
empty-c3c18-1.lif    Image004               219                    0            0.000
empty-c3c18-1.lif    Image005               355                    1            0.003
empty-c3c18-1.lif    Image006               604                    0            0.000
empty-c3c18-1.lif    Image007               434                   

,lif_file,lif_stem,series_index,series_name,protoplast_count,nuclei_count_manual,nuclei_per_cell,click_coordinates
0,empty-c3c18-1.lif,empty-c3c18-1,0,Image001,171,0,0.000,[]
1,empty-c3c18-1.lif,empty-c3c18-1,1,Image002,220,0,0.000,[]
2,empty-c3c18-1.lif,empty-c3c18-1,2,Image003,250,0,0.000,[]
3,empty-c3c18-1.lif,empty-c3c18-1,3,Image004,219,0,0.000,[]
4,empty-c3c18-1.lif,empty-c3c18-1,4,Image005,355,1,0.003,"[(np.float64(946.7237476808908), np.float64(79..."
5,empty-c3c18-1.lif,empty-c3c18-1,5,Image006,604,0,0.000,[]
6,empty-c3c18-1.lif,empty-c3c18-1,6,Image007,434,2,0.005,"[(np.float64(197.00946196660493), np.float64(9..."
7,empty-c3c18-1.lif,empty-c3c18-1,7,Image008,823,1,0.001,"[(np.float64(360.5834879406309), np.float64(34..."
8,empty-c3c18-1.lif,empty-c3c18-1,8,Image009,299,0,0.000,[]
9,empty-c3c18-1.lif,empty-c3c18-1,9,Image010,219,0,0.000,[]


## 7 · Load Saved Results (Fresh Session)

In [ ]:
# ── Load previously saved results (fresh session) ────────────────────────────
import pandas as pd
from pathlib import Path

csv_path = Path(OUTPUT_DIR) / "manual_counts.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print(f"\nSummary:")
    print(df[["protoplast_count","nuclei_count_manual","nuclei_per_cell"]].describe())
else:
    print(f"No saved results found at {csv_path}")

No saved results found at countedPEG1/manual_counts.csv


/Users/mark/anaconda3/envs/biocircuits/lib/python3.11/site-packages/cellpose/dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


: 